In [1]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

In [2]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def audit_df(df, text_col, label_col=None):
    print("Rows:", len(df))
    print("Columns:", list(df.columns))
    print("\nMissing values:")
    print(df.isna().sum())
    lengths = df[text_col].astype(str).str.len()
    print(lengths.describe())
    print("\nDuplicate text rows:", df.duplicated(subset=[text_col]).sum())
    print(df[label_col].value_counts(dropna=False))


def light_clean_df(df, text_col):
    df = df.copy()
    df[text_col] = df[text_col].astype(str).apply(clean_text)
    df = df[df[text_col].str.len() > 0]
    df = df.drop_duplicates(subset=[text_col])
    return df.reset_index(drop=True)


In [3]:
banking_test = pd.read_csv("data/raw/banking77_test.csv")
banking_train = pd.read_csv("data/raw/banking77_train.csv")
phrasebank = pd.read_csv("data/raw/all-data.csv", encoding = "latin-1",header=None,             # Correctly stops pandas from using your first data row as a header
    names=["label", "text"])  # this file has some non-UTF-8 chars, so specify encoding

frimi = pd.read_csv("data/raw/FriMi_FinTech_raw_reviews.csv")
keells = pd.read_csv("data/raw/Keells_Supermarket_raw_reviews.csv")
peoplespay = pd.read_csv("data/raw/Peoples_Pay_raw_reviews.csv")
combank = pd.read_csv("data/raw/ComBank_Digital_raw_reviews.csv")
flexpay = pd.read_csv("data/raw/FlexPay_BOC_raw_reviews.csv")
HNB = pd.read_csv("data/raw/HNB_DigitalBanking_raw_reviews.csv")
NDB = pd.read_csv("data/raw/NDB_MobileBanking_raw_reviews.csv")
NSB = pd.read_csv("data/raw/NSBPay_raw_reviews.csv")



In [4]:
audit_df(banking_test, text_col="text", label_col="label")
audit_df(banking_train, text_col="text", label_col="label")


Rows: 3076
Columns: ['text', 'label', 'label_text']

Missing values:
text          0
label         0
label_text    0
dtype: int64
count    3076.000000
mean       54.276983
std        34.657985
min        13.000000
25%        35.000000
50%        45.000000
75%        60.000000
max       368.000000
Name: text, dtype: float64

Duplicate text rows: 0
label
11    40
63    40
50    40
64    40
7     40
      ..
73    40
3     39
48    39
66    39
33    39
Name: count, Length: 77, dtype: int64
Rows: 9993
Columns: ['text', 'label', 'label_text']

Missing values:
text          0
label         0
label_text    0
dtype: int64
count    9993.000000
mean       59.501351
std        40.873384
min        13.000000
25%        36.000000
50%        47.000000
75%        64.000000
max       433.000000
Name: text, dtype: float64

Duplicate text rows: 0
label
15    187
28    182
6     181
75    180
19    177
     ... 
41     82
18     61
10     59
72     41
23     35
Name: count, Length: 77, dtype: int64


In [5]:
audit_df(phrasebank, text_col="text", label_col="label")


Rows: 4846
Columns: ['label', 'text']

Missing values:
label    0
text     0
dtype: int64
count    4846.000000
mean      128.132068
std        56.526180
min         9.000000
25%        84.000000
50%       119.000000
75%       163.000000
max       315.000000
Name: text, dtype: float64

Duplicate text rows: 8
label
neutral     2879
positive    1363
negative     604
Name: count, dtype: int64


In [6]:
audit_df(frimi, text_col="review_text", label_col="star_rating")
audit_df(keells, text_col="review_text", label_col="star_rating")
audit_df(peoplespay, text_col="review_text", label_col="star_rating")
audit_df(combank, text_col="review_text", label_col="star_rating")
audit_df(flexpay, text_col="review_text", label_col="star_rating")
audit_df(NDB, text_col="review_text", label_col="star_rating")
audit_df(HNB, text_col="review_text", label_col="star_rating")
audit_df(NSB, text_col="review_text", label_col="star_rating")


Rows: 6555
Columns: ['source_app', 'review_text', 'star_rating', 'recorded_at', 'app_version', 'source_row_id', 'normalized_text']

Missing values:
source_app           0
review_text          0
star_rating          0
recorded_at          0
app_version        903
source_row_id        0
normalized_text      0
dtype: int64
count    6555.000000
mean      138.070175
std       127.234730
min        16.000000
25%        49.000000
50%        92.000000
75%       185.000000
max      1795.000000
Name: review_text, dtype: float64

Duplicate text rows: 4319
star_rating
1    3282
5    2097
4     460
3     396
2     320
Name: count, dtype: int64
Rows: 960
Columns: ['source_app', 'review_text', 'star_rating', 'recorded_at', 'app_version', 'source_row_id', 'normalized_text']

Missing values:
source_app           0
review_text          0
star_rating          0
recorded_at          0
app_version        144
source_row_id        0
normalized_text      0
dtype: int64
count    960.000000
mean     123.598958


In [7]:
import unicodedata
clean_banking_test  = light_clean_df(banking_test, text_col="text")
clean_banking_train = light_clean_df(banking_train, text_col="text")
clean_phrasebank    = light_clean_df(phrasebank, text_col="text")

clean_frimi      = light_clean_df(frimi, text_col="review_text")
clean_keells     = light_clean_df(keells, text_col="review_text")
clean_peoplespay = light_clean_df(peoplespay, text_col="review_text")
clean_combank    = light_clean_df(combank, text_col="review_text")
clean_flexpay    = light_clean_df(flexpay, text_col="review_text")
clean_HNB      = light_clean_df(HNB, text_col="review_text")
clean_NSB      = light_clean_df(NSB, text_col="review_text")
clean_NDB     = light_clean_df(NDB, text_col="review_text")



In [8]:
print("UNIQUE LABELS IN FINANCIAL PHRASEBANK")
print(clean_phrasebank["label"].unique())

print("SAMPLE OF UNIQUE INTENTS IN BANKING77")
print(list(clean_banking_train["label_text"].unique()))


UNIQUE LABELS IN FINANCIAL PHRASEBANK
['neutral' 'negative' 'positive']
SAMPLE OF UNIQUE INTENTS IN BANKING77
['card_arrival', 'card_linking', 'exchange_rate', 'card_payment_wrong_exchange_rate', 'extra_charge_on_statement', 'pending_cash_withdrawal', 'fiat_currency_support', 'card_delivery_estimate', 'automatic_top_up', 'card_not_working', 'exchange_via_app', 'lost_or_stolen_card', 'age_limit', 'pin_blocked', 'contactless_not_working', 'top_up_by_bank_transfer_charge', 'pending_top_up', 'cancel_transfer', 'top_up_limits', 'wrong_amount_of_cash_received', 'card_payment_fee_charged', 'transfer_not_received_by_recipient', 'supported_cards_and_currencies', 'getting_virtual_card', 'card_acceptance', 'top_up_reverted', 'balance_not_updated_after_cheque_or_cash_deposit', 'card_payment_not_recognised', 'edit_personal_details', 'why_verify_identity', 'unable_to_verify_identity', 'get_physical_card', 'visa_or_mastercard', 'topping_up_by_card', 'disposable_card_limits', 'compromised_card', 'atm_

In [9]:
# 1. Preview Banking Datasets
print("CLEANED BANKING TEST PREVIEW")
display(clean_banking_test.head(3))

print("CLEANED BANKING TRAIN PREVIEW")
display(clean_banking_train.head(3))

# 2. Preview Phrasebank Dataset
print("CLEANED FINANCIAL PHRASEBANK PREVIEW")
display(clean_phrasebank.head(3))


CLEANED BANKING TEST PREVIEW


,text,label,label_text
0,How do I locate my card?,11,card_arrival
1,"I still have not received my new card, I order...",11,card_arrival
2,I ordered a card but it has not arrived. Help ...,11,card_arrival


CLEANED BANKING TRAIN PREVIEW


,text,label,label_text
0,I am still waiting on my card?,11,card_arrival
1,What can I do if my card still hasn't arrived ...,11,card_arrival
2,I have been waiting over a week. Is the card s...,11,card_arrival


CLEANED FINANCIAL PHRASEBANK PREVIEW


,label,text
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...


In [10]:

clean_banking_test.to_csv("data/processed/banking77_test_clean.csv", index=False)
clean_banking_train.to_csv("data/processed/banking77_train_clean.csv", index=False)

clean_phrasebank.to_csv("data/processed/financial_phrasebank_clean.csv", index=False)




In [ ]:
clean_bank_reviews = pd.concat(
    [
        clean_frimi,
        clean_keells,
        clean_peoplespay,
        clean_combank,
        clean_flexpay,
        clean_HNB,
        clean_NSB,
        clean_NDB,
    ],
    ignore_index=True,
)

if "normalized_text" not in clean_bank_reviews.columns:
    clean_bank_reviews["normalized_text"] = clean_bank_reviews["review_text"].apply(clean_text)

clean_bank_reviews = clean_bank_reviews.drop_duplicates(
    subset=["normalized_text", "recorded_at"],
    keep="first",
).reset_index(drop=True)
clean_bank_reviews["source_row_id"] = range(1, len(clean_bank_reviews) + 1)

clean_bank_reviews.to_csv(
    "data/processed/scraped_reviews_clean_combined.csv",
    index=False,
)

print(
    "Saved cleaned bank reviews: "
    f"data/processed/scraped_reviews_combined.csv)"
)
print("Columns:", list(clean_bank_reviews.columns))

Saved cleaned bank reviews: data/processed/scraped_reviews_combined.csv (12,523 rows)
Columns: ['source_app', 'review_text', 'star_rating', 'recorded_at', 'app_version', 'source_row_id', 'normalized_text']
